In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

In [2]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split, GridSearchCV

In [3]:
# ============================================================
# STEP 1: LOAD DATA
# ============================================================
import pandas as pd

file_path = "dataset/NIFTY_50_COMPANIES.csv" 
df = pd.read_csv(file_path)

print("="*60)
print("BASIC INFORMATION")
print("="*60)

# Shape
print(f"Shape (Rows, Columns): {df.shape}")

# Column names
print("\nColumns:")
print(df.columns.tolist())

# Data types
print("\nData Types:")
print(df.dtypes)

# ============================================================
# STEP 2: DATE INFORMATION
# ============================================================
print("\n" + "="*60)
print("DATE INFORMATION")
print("="*60)

df['Date'] = pd.to_datetime(df['Date'], errors='coerce')

print(f"Start Date : {df['Date'].min()}")
print(f"End Date   : {df['Date'].max()}")
print(f"Total Unique Dates : {df['Date'].nunique()}")

# ============================================================
# STEP 3: TICKER INFORMATION
# ============================================================
print("\n" + "="*60)
print("TICKER INFORMATION")
print("="*60)

if 'Ticker' in df.columns:
    print(f"Total Unique Tickers : {df['Ticker'].nunique()}")
    print("Ticker List:")
    print(df['Ticker'].unique())

# ============================================================
# STEP 4: MISSING VALUES
# ============================================================
print("\n" + "="*60)
print("MISSING VALUES")
print("="*60)

missing = df.isnull().sum()
print(missing[missing > 0])

# ============================================================
# STEP 5: NUMERICAL SUMMARY
# ============================================================
print("\n" + "="*60)
print("NUMERICAL SUMMARY")
print("="*60)

print(df.describe())

# ============================================================
# STEP 6: SAMPLE DATA
# ============================================================
print("\n" + "="*60)
print("FIRST 5 ROWS")
print("="*60)
print(df.head())

print("\n" + "="*60)
print("LAST 5 ROWS")
print("="*60)
print(df.tail())

BASIC INFORMATION
Shape (Rows, Columns): (304543, 19)

Columns:
['Date', 'Adj Close', 'Close', 'High', 'Low', 'Open', 'Volume', 'Ticker', 'SMA_20', 'SMA_50', 'EMA_12', 'EMA_26', 'MACD', 'Signal_Line', 'RSI_14', 'BB_Mid', 'BB_Upper', 'BB_Lower', 'Daily_Return_%']

Data Types:
Date                  str
Adj Close         float64
Close             float64
High              float64
Low               float64
Open              float64
Volume              int64
Ticker                str
SMA_20            float64
SMA_50            float64
EMA_12            float64
EMA_26            float64
MACD              float64
Signal_Line       float64
RSI_14            float64
BB_Mid            float64
BB_Upper          float64
BB_Lower          float64
Daily_Return_%    float64
dtype: object

DATE INFORMATION
Start Date : 1997-07-01 00:00:00
End Date   : 2025-11-11 00:00:00
Total Unique Dates : 7109

TICKER INFORMATION
Total Unique Tickers : 51
Ticker List:
<StringArray>
[  'RELIANCE.NS',        'TCS.NS'

In [4]:
# ============================================================
# STEP 2: DATA CLEANING
# ============================================================
print("\n" + "=" * 60)
print("STEP 2: DATA CLEANING")
print("=" * 60)

before = len(df)
print(f"Shape before: {df.shape}")
null_counts = df.isnull().sum()
print(f"Null counts:\n{null_counts[null_counts > 0].to_string()}")
df = df.dropna()
print(f"Rows dropped (NaN from indicator lookback): {before - len(df)}")
print(f"Shape after : {df.shape}")


STEP 2: DATA CLEANING
Shape before: (304543, 19)
Null counts:
SMA_20             969
SMA_50            2499
RSI_14            2697
BB_Mid             969
BB_Upper           969
BB_Lower           969
Daily_Return_%      51
Rows dropped (NaN from indicator lookback): 4533
Shape after : (300010, 19)


In [5]:
# ============================================================
# STEP 3: FEATURE ENGINEERING
# ============================================================

print("\n" + "=" * 60)
print("STEP 3: FEATURE ENGINEERING")
print("=" * 60)

import numpy as np

# Ensure sorted properly
df = df.sort_values(['Ticker', 'Date']).reset_index(drop=True)

# ------------------------------------------------------------
# FEATURE ENGINEERING PER TICKER (PROFESSIONAL METHOD)
# ------------------------------------------------------------

# 1️⃣ Daily Return (if not already correct)
df['Return'] = df.groupby('Ticker')['Adj Close'].pct_change()

# 2️⃣ Lag Features
df['Return_Lag1'] = df.groupby('Ticker')['Return'].shift(1)
df['Return_Lag2'] = df.groupby('Ticker')['Return'].shift(2)
df['Return_Lag3'] = df.groupby('Ticker')['Return'].shift(3)

# 3️⃣ Rolling Volatility
df['Volatility_10'] = (
    df.groupby('Ticker')['Return']
      .rolling(10)
      .std()
      .reset_index(level=0, drop=True)
)

# 4️⃣ Trend Ratios
df['SMA_ratio'] = df['Close'] / df['SMA_20']
df['EMA_ratio'] = df['Close'] / df['EMA_12']

# 5️⃣ Momentum Features
df['MACD_diff'] = df['MACD'] - df['Signal_Line']
df['RSI_norm']  = df['RSI_14'] / 100

# 6️⃣ Bollinger Band Position
bb_width = (df['BB_Upper'] - df['BB_Lower']).replace(0, np.nan)
df['BB_position'] = (df['Close'] - df['BB_Lower']) / bb_width

# ------------------------------------------------------------
# Show Added Features
# ------------------------------------------------------------

eng_features = [
    'Return',
    'Return_Lag1',
    'Return_Lag2',
    'Return_Lag3',
    'Volatility_10',
    'SMA_ratio',
    'EMA_ratio',
    'MACD_diff',
    'RSI_norm',
    'BB_position'
]

print("Features added:")
print(eng_features)

print("\nFeature Statistics:")
print(df[eng_features].describe().round(4))


STEP 3: FEATURE ENGINEERING
Features added:
['Return', 'Return_Lag1', 'Return_Lag2', 'Return_Lag3', 'Volatility_10', 'SMA_ratio', 'EMA_ratio', 'MACD_diff', 'RSI_norm', 'BB_position']

Feature Statistics:
            Return  Return_Lag1  Return_Lag2  Return_Lag3  Volatility_10  \
count  299959.0000  299908.0000  299857.0000  299806.0000    299500.0000   
mean        0.0020       0.0020       0.0020       0.0020         0.0237   
std         0.2140       0.2141       0.2141       0.2141         0.2133   
min        -0.9909      -0.9909      -0.9909      -0.9909         0.0000   
25%        -0.0102      -0.0102      -0.0102      -0.0102         0.0125   
50%         0.0001       0.0001       0.0001       0.0001         0.0172   
75%         0.0114       0.0114       0.0114       0.0114         0.0242   
max       108.4930     108.4930     108.4930     108.4930        34.3460   

         SMA_ratio    EMA_ratio    MACD_diff     RSI_norm  BB_position  
count  300010.0000  300010.0000  3000

In [6]:
# ============================================================
# STEP 4: CREATE TARGET LABEL (PROFESSIONAL VERSION)
# ============================================================

print("\n" + "=" * 60)
print("STEP 4: CREATE TARGET LABEL")
print("=" * 60)

# Ensure sorted properly
df = df.sort_values(['Ticker', 'Date']).reset_index(drop=True)

# ------------------------------------------------------------
# 1️⃣ Create Future Return per ticker
# ------------------------------------------------------------

df['Future_Return'] = df.groupby('Ticker')['Return'].shift(-1)

# ------------------------------------------------------------
# 2️⃣ Define Signal using vectorized logic
# ------------------------------------------------------------

threshold = 0.005  # 0.5%

df['Signal'] = 0
df.loc[df['Future_Return'] >  threshold, 'Signal'] =  1
df.loc[df['Future_Return'] < -threshold, 'Signal'] = -1

# ------------------------------------------------------------
# 3️⃣ Drop rows where Future_Return is NaN (last row per stock)
# ------------------------------------------------------------

df = df.dropna(subset=['Future_Return'])

# ------------------------------------------------------------
# 4️⃣ Print Class Distribution
# ------------------------------------------------------------

print(f"Threshold: ±{threshold*100:.2f}%  |  BUY=1, HOLD=0, SELL=-1\n")

counts = df['Signal'].value_counts().sort_index()

label_map = {1:'BUY', 0:'HOLD', -1:'SELL'}

for k in counts.index:
    v = counts[k]
    print(f"{label_map[k]:5s} ({k:+d}): {v:7d}  ({v/len(df)*100:.2f}%)")


STEP 4: CREATE TARGET LABEL
Threshold: ±0.50%  |  BUY=1, HOLD=0, SELL=-1

SELL  (-1):  107601  (35.87%)
HOLD  (+0):   79875  (26.63%)
BUY   (+1):  112483  (37.50%)


In [7]:
# ============================================================
# STEP 5: SELECT FINAL FEATURES
# ============================================================

print("\n" + "=" * 60)
print("STEP 5: SELECT FINAL FEATURES")
print("=" * 60)

# Ensure properly sorted
df = df.sort_values(['Ticker', 'Date']).reset_index(drop=True)

# ------------------------------------------------------------
# Final Feature List
# ------------------------------------------------------------

features = [
    'Return_Lag1',
    'Return_Lag2',
    'Return_Lag3',
    'Volatility_10',
    'SMA_ratio',
    'EMA_ratio',
    'MACD_diff',
    'RSI_norm',
    'BB_position'
]

# ------------------------------------------------------------
# Remove rows with missing values in features or target
# ------------------------------------------------------------

df = df.dropna(subset=features + ['Signal'])

# ------------------------------------------------------------
# Define X and y
# ------------------------------------------------------------

X = df[features]
y = df['Signal']

print(f"Selected Features ({len(features)}):")
print(features)

print(f"\nFinal Dataset Shape:")
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

# Check for remaining NaNs
print("\nRemaining NaNs in X:")
print(X.isnull().sum().sum())


STEP 5: SELECT FINAL FEATURES
Selected Features (9):
['Return_Lag1', 'Return_Lag2', 'Return_Lag3', 'Volatility_10', 'SMA_ratio', 'EMA_ratio', 'MACD_diff', 'RSI_norm', 'BB_position']

Final Dataset Shape:
X shape: (299449, 9)
y shape: (299449,)

Remaining NaNs in X:
0


In [8]:
# ============================================================
# STEP 6: WALK-FORWARD VALIDATION (TimeSeriesSplit)
# ============================================================

print("\n" + "=" * 60)
print("STEP 6: WALK-FORWARD VALIDATION (TimeSeriesSplit)")
print("=" * 60)

from sklearn.model_selection import TimeSeriesSplit
import numpy as np

# Ensure sorted globally by Date
df = df.sort_values(['Date', 'Ticker']).reset_index(drop=True)

X = df[features]
y = df['Signal']

tscv = TimeSeriesSplit(n_splits=5)

fold = 1

for train_index, test_index in tscv.split(X):

    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    d_train = df['Date'].iloc[train_index]
    d_test  = df['Date'].iloc[test_index]

    print(f"\n📊 Fold {fold}")
    print(f"Train: {len(X_train):6,} rows  "
          f"{d_train.min().date()} → {d_train.max().date()}")

    print(f"Test : {len(X_test):6,} rows  "
          f"{d_test.min().date()} → {d_test.max().date()}")

    fold += 1


STEP 6: WALK-FORWARD VALIDATION (TimeSeriesSplit)

📊 Fold 1
Train: 49,909 rows  1997-09-22 → 2004-11-09
Test : 49,908 rows  2004-11-09 → 2009-06-10

📊 Fold 2
Train: 99,817 rows  1997-09-22 → 2009-06-10
Test : 49,908 rows  2009-06-10 → 2013-09-05

📊 Fold 3
Train: 149,725 rows  1997-09-22 → 2013-09-05
Test : 49,908 rows  2013-09-05 → 2017-11-29

📊 Fold 4
Train: 199,633 rows  1997-09-22 → 2017-11-29
Test : 49,908 rows  2017-11-29 → 2021-11-26

📊 Fold 5
Train: 249,541 rows  1997-09-22 → 2021-11-26
Test : 49,908 rows  2021-11-26 → 2025-11-10


In [9]:
# ============================================================
# STEP 7: FEATURE SCALING
# ============================================================

print("\n" + "=" * 60)
print("STEP 7: FEATURE SCALING (StandardScaler)")
print("=" * 60)

from sklearn.preprocessing import StandardScaler
import numpy as np

# Create scaler object
scaler = StandardScaler()

# ------------------------------------------------------------
# IMPORTANT:
# fit() ONLY on TRAIN data
# transform() on TRAIN and TEST
# ------------------------------------------------------------

X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print("✔ fit() on TRAIN only")
print("✔ transform() on TEST")
print("✔ No data leakage")

# ------------------------------------------------------------
# Verification
# ------------------------------------------------------------

print("\nPost-Scaling Check (Train Data):")
print(f"Mean ≈ {np.mean(X_train_sc):.6f}   (target ≈ 0)")
print(f"Std  ≈ {np.std(X_train_sc):.6f}    (target ≈ 1)")

print("\nShapes After Scaling:")
print(f"X_train_sc shape: {X_train_sc.shape}")
print(f"X_test_sc  shape: {X_test_sc.shape}")


STEP 7: FEATURE SCALING (StandardScaler)
✔ fit() on TRAIN only
✔ transform() on TEST
✔ No data leakage

Post-Scaling Check (Train Data):
Mean ≈ 0.000000   (target ≈ 0)
Std  ≈ 1.000000    (target ≈ 1)

Shapes After Scaling:
X_train_sc shape: (249541, 9)
X_test_sc  shape: (49908, 9)


In [10]:
# ============================================================
# STEP 8: TRAIN MULTIPLE SUPERVISED MODELS
# ============================================================

print("\n" + "=" * 60)
print("STEP 8: TRAIN MULTIPLE SUPERVISED MODELS")
print("=" * 60)

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    AdaBoostClassifier,
    HistGradientBoostingClassifier
)
from sklearn.metrics import accuracy_score
import pandas as pd
import numpy as np

models = {

    # Linear
    "Logistic Regression":
        LogisticRegression(max_iter=1000, class_weight='balanced'),

    # Distance
    "KNN":
        KNeighborsClassifier(n_neighbors=5),

    # SVM
    "SVM (Linear)":
        SVC(class_weight='balanced', max_iter=5000),

    "SVM (RBF)":
        SVC(class_weight='balanced', max_iter=5000),

    # Tree
    "Decision Tree":
        DecisionTreeClassifier(max_depth=6, class_weight='balanced'),

    "Random Forest":
        RandomForestClassifier(
            n_estimators=200,
            max_depth=6,
            class_weight='balanced',
            random_state=42,
            n_jobs=-1
        ),

    # Boosting
    "Gradient Boosting":
        GradientBoostingClassifier(),

    "AdaBoost":
        AdaBoostClassifier(),

    "HistGradientBoosting":
        HistGradientBoostingClassifier()
}

results = {}

for name, model in models.items():

    print(f"\nTraining {name}...")

    if name in ["Decision Tree", "Random Forest",
                "Gradient Boosting", "AdaBoost",
                "HistGradientBoosting"]:
        
        model.fit(X_train, y_train)
        pred = model.predict(X_test)
    else:
        model.fit(X_train_sc, y_train)
        pred = model.predict(X_test_sc)

    acc = accuracy_score(y_test, pred)
    results[name] = acc

    print(f"{name} Accuracy: {acc:.4f}")

# ------------------------------------------------------------
# Final Comparison
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("MODEL COMPARISON (Accuracy)")
print("=" * 60)

for name, score in sorted(results.items(), key=lambda x: x[1], reverse=True):
    print(f"{name:25s}: {score:.4f}")


STEP 8: TRAIN MULTIPLE SUPERVISED MODELS

Training Logistic Regression...
Logistic Regression Accuracy: 0.3462

Training KNN...
KNN Accuracy: 0.3371

Training SVM (Linear)...
SVM (Linear) Accuracy: 0.3240

Training SVM (RBF)...
SVM (RBF) Accuracy: 0.3240

Training Decision Tree...
Decision Tree Accuracy: 0.3516

Training Random Forest...
Random Forest Accuracy: 0.3543

Training Gradient Boosting...
Gradient Boosting Accuracy: 0.3627

Training AdaBoost...
AdaBoost Accuracy: 0.3463

Training HistGradientBoosting...
HistGradientBoosting Accuracy: 0.3619

MODEL COMPARISON (Accuracy)
Gradient Boosting        : 0.3627
HistGradientBoosting     : 0.3619
Random Forest            : 0.3543
Decision Tree            : 0.3516
AdaBoost                 : 0.3463
Logistic Regression      : 0.3462
KNN                      : 0.3371
SVM (Linear)             : 0.3240
SVM (RBF)                : 0.3240


In [11]:
# ============================================================
# SELECT BEST MODEL + RETRAIN + EVALUATE
# ============================================================

from sklearn.metrics import classification_report, confusion_matrix

# ------------------------------------------------------------
# STEP 1: Find Best Model
# ------------------------------------------------------------

best_model_name = max(results, key=results.get)
print(f"\nBest Model: {best_model_name}")

best_model = models[best_model_name]

# ------------------------------------------------------------
# STEP 2: Retrain Best Model Properly
# ------------------------------------------------------------

tree_based_models = [
    "Decision Tree",
    "Random Forest",
    "Gradient Boosting",
    "AdaBoost",
    "HistGradientBoosting"
]

if best_model_name in tree_based_models:
    best_model.fit(X_train, y_train)
    pred = best_model.predict(X_test)
else:
    best_model.fit(X_train_sc, y_train)
    pred = best_model.predict(X_test_sc)

# ------------------------------------------------------------
# STEP 3: Evaluate
# ------------------------------------------------------------

print("\nClassification Report:\n")

print(classification_report(
    y_test,
    pred,
    target_names=['SELL(-1)', 'HOLD(0)', 'BUY(+1)'],
    labels=[-1, 0, 1]
))

cm = confusion_matrix(y_test, pred, labels=[-1, 0, 1])

print("Confusion Matrix (rows=Actual, cols=Predicted):")
print(f"{'':12s} SELL   HOLD   BUY")

for i, lab in enumerate(['SELL', 'HOLD', 'BUY']):
    print(f"  {lab:8s} {cm[i]}")


Best Model: Gradient Boosting

Classification Report:

              precision    recall  f1-score   support

    SELL(-1)       0.35      0.14      0.20     16271
     HOLD(0)       0.38      0.29      0.33     16080
     BUY(+1)       0.36      0.64      0.46     17557

    accuracy                           0.36     49908
   macro avg       0.36      0.36      0.33     49908
weighted avg       0.36      0.36      0.33     49908

Confusion Matrix (rows=Actual, cols=Predicted):
             SELL   HOLD   BUY
  SELL     [ 2241  3648 10382]
  HOLD     [1885 4627 9568]
  BUY      [ 2308  4016 11233]


In [16]:
# ------------------------------------------------------------
# Compare Models
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("MODEL COMPARISON (Accuracy)")
print("=" * 60)

for name, score in sorted(results.items(), key=lambda x: x[1], reverse=True):
    print(f"{name:25s}: {score:.4f}")

# ------------------------------------------------------------
# SELECT BEST MODEL
# ------------------------------------------------------------

best_model_name = max(results, key=results.get)
print(f"\nBest Model Selected: {best_model_name}")

best_model = models[best_model_name]

# Retrain best model properly
tree_models = [
    "Decision Tree",
    "Random Forest",
    "Gradient Boosting",
    "AdaBoost",
    "HistGradientBoosting"
]

if best_model_name in tree_models:
    best_model.fit(X_train, y_train)
    best_pred = best_model.predict(X_test)
else:
    best_model.fit(X_train_sc, y_train)
    best_pred = best_model.predict(X_test_sc)


MODEL COMPARISON (Accuracy)
Gradient Boosting        : 0.3627
HistGradientBoosting     : 0.3619
Random Forest            : 0.3543
Decision Tree            : 0.3516
AdaBoost                 : 0.3463
Logistic Regression      : 0.3462
KNN                      : 0.3371
SVM (Linear)             : 0.3240
SVM (RBF)                : 0.3240

Best Model Selected: Gradient Boosting


In [12]:
import joblib

joblib.dump(best_model, "best_model.pkl")

print("Best model saved successfully!")

Best model saved successfully!


In [ ]:
print("\n" + "=" * 60)
print("STEP 10: BACKTEST + RISK METRICS")
print("=" * 60)

# Prepare test dataframe using test_index from the last fold of TimeSeriesSplit
df_test = df.iloc[test_index].copy().reset_index(drop=True)

# Align predictions
best_pred_series = pd.Series(best_pred).reset_index(drop=True)
df_test = df_test.iloc[:len(best_pred_series)].copy()
df_test['Predicted_Signal'] = best_pred_series.values

# Strategy return
df_test['Strategy_Return'] = (
    df_test['Predicted_Signal'] *
    df_test['Future_Return']
)

# Transaction cost
tc = 0.001

df_test['Trade'] = (
    df_test['Predicted_Signal']
    .diff()
    .abs()
    .fillna(0)
)

df_test['Strategy_Return'] -= tc * df_test['Trade']

# Cumulative returns
df_test['Cum_Strategy'] = (1 + df_test['Strategy_Return']).cumprod()
df_test['Cum_Market']   = (1 + df_test['Future_Return']).cumprod()

# Risk metrics
if df_test['Strategy_Return'].std() != 0:
    sharpe = (
        df_test['Strategy_Return'].mean() /
        df_test['Strategy_Return'].std()
    ) * np.sqrt(252)
else:
    sharpe = 0

roll_max = df_test['Cum_Strategy'].cummax()
drawdown = df_test['Cum_Strategy'] / roll_max - 1
max_dd   = drawdown.min()

final_s = df_test['Cum_Strategy'].iloc[-1]
final_m = df_test['Cum_Market'].iloc[-1]

# ------------------------------------------------------------
# Print Results
# ------------------------------------------------------------

print(f"  Strategy Total Return : {(final_s - 1) * 100:+.2f}%")
print(f"  Market  Total Return  : {(final_m - 1) * 100:+.2f}%")
print(f"  Sharpe Ratio          : {sharpe:.4f}")
print(f"  Max Drawdown          : {max_dd * 100:.2f}%")
print(f"  Transaction Cost      : {tc * 100:.1f}% per trade side")


STEP 10: BACKTEST + RISK METRICS


NameError: name 'split' is not defined

In [ ]:
# ============================================================
# SINGLE-TICKER BACKTEST (RELIANCE ONLY)
# ============================================================

import numpy as np
import pandas as pd

ticker_name = "RELIANCE.NS"

print("\nRunning single-ticker backtest for:", ticker_name)

# ------------------------------------------------------------
# Filter only RELIANCE
# ------------------------------------------------------------

df_rel = df[df['Ticker'] == ticker_name].copy()
df_rel = df_rel.sort_values('Date').reset_index(drop=True)

print(f"Rows for {ticker_name}: {len(df_rel)}")

# ------------------------------------------------------------
# Create Features & Target
# ------------------------------------------------------------

feature_cols = features  # same features used before

X_rel = df_rel[feature_cols]
y_rel = df_rel['Signal']

# ------------------------------------------------------------
# Train-Test Split (Time-Based)
# ------------------------------------------------------------

split_idx = int(len(df_rel) * 0.8)

X_train_r = X_rel.iloc[:split_idx]
X_test_r  = X_rel.iloc[split_idx:]

y_train_r = y_rel.iloc[:split_idx]
y_test_r  = y_rel.iloc[split_idx:]

# ------------------------------------------------------------
# Scaling (if needed)
# ------------------------------------------------------------

from sklearn.preprocessing import StandardScaler

scaler_r = StandardScaler()

X_train_r_sc = scaler_r.fit_transform(X_train_r)
X_test_r_sc  = scaler_r.transform(X_test_r)

# ------------------------------------------------------------
# Train Best Model (Random Forest)
# ------------------------------------------------------------

from sklearn.ensemble import RandomForestClassifier

model_r = RandomForestClassifier(
    n_estimators=200,
    max_depth=6,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

model_r.fit(X_train_r, y_train_r)
pred_r = model_r.predict(X_test_r)

# ------------------------------------------------------------
# Backtest
# ------------------------------------------------------------

df_test_r = df_rel.iloc[split_idx:].copy().reset_index(drop=True)
df_test_r['Predicted_Signal'] = pred_r

df_test_r['Strategy_Return'] = (
    df_test_r['Predicted_Signal'] *
    df_test_r['Future_Return']
)

# Transaction cost
tc = 0.001

df_test_r['Trade'] = (
    df_test_r['Predicted_Signal']
    .diff()
    .abs()
    .fillna(0)
)

df_test_r['Strategy_Return'] -= tc * df_test_r['Trade']

# Cumulative returns
df_test_r['Cum_Strategy'] = (1 + df_test_r['Strategy_Return']).cumprod()
df_test_r['Cum_Market']   = (1 + df_test_r['Future_Return']).cumprod()

# Risk metrics
if df_test_r['Strategy_Return'].std() != 0:
    sharpe_r = (
        df_test_r['Strategy_Return'].mean() /
        df_test_r['Strategy_Return'].std()
    ) * np.sqrt(252)
else:
    sharpe_r = 0

roll_max_r = df_test_r['Cum_Strategy'].cummax()
drawdown_r = df_test_r['Cum_Strategy'] / roll_max_r - 1
max_dd_r   = drawdown_r.min()

final_s_r = df_test_r['Cum_Strategy'].iloc[-1]
final_m_r = df_test_r['Cum_Market'].iloc[-1]

# ------------------------------------------------------------
# Print Results
# ------------------------------------------------------------

print("\n" + "=" * 50)
print(f"RELIANCE BACKTEST RESULTS")
print("=" * 50)

print(f"Strategy Return : {(final_s_r-1)*100:+.2f}%")
print(f"Market Return   : {(final_m_r-1)*100:+.2f}%")
print(f"Sharpe Ratio    : {sharpe_r:.4f}")
print(f"Max Drawdown    : {max_dd_r*100:.2f}%")